In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import optuna
from sklearn.datasets import fetch_openml


In [2]:
# Загружаем данные
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame.copy()

# Целевая переменная и признаки
X = df.drop(columns=['survived'])
y = df['survived'].astype('int')  # Преобразуем в целочисленный формат

/Users/stureiko/miniforge3/envs/otus/lib/python3.10/site-packages/sklearn/datasets/_openml.py:1022: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


In [3]:
# Выделяем числовые и категориальные признаки
num_cols = X.select_dtypes(include=['float64', 'int']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

In [4]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [5]:
# Препроцессинг
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

In [6]:
# Оптимизация гиперпараметров с помощью Optuna
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }

    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42, **params))
    ])

    scores = cross_val_score(clf, X_train, y_train, cv=3, scoring='accuracy')
    return scores.mean()


In [7]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Лучшие гиперпараметры:")
print(study.best_params)

[I 2025-04-19 08:52:51,057] A new study created in memory with name: no-name-684ac180-ed3b-48f3-a1ca-09e709ed673a
[I 2025-04-19 08:52:52,145] Trial 0 finished with value: 0.9551098376313276 and parameters: {'n_estimators': 183, 'max_depth': 6, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.9551098376313276.
[I 2025-04-19 08:52:52,385] Trial 1 finished with value: 0.617956064947469 and parameters: {'n_estimators': 114, 'max_depth': 17, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.9551098376313276.
[I 2025-04-19 08:52:52,801] Trial 2 finished with value: 0.617956064947469 and parameters: {'n_estimators': 251, 'max_depth': 3, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.9551098376313276.
[I 2025-04-19 08:52:53,309] Trial 3 finished with value: 0.617956064947469 and parameters: {'n_estimators': 298, 'max_depth': 6, 'min_sampl

Лучшие гиперпараметры:
{'n_estimators': 130, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_features': None}


In [8]:
# Финальная модель с лучшими параметрами
best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, **study.best_params))
])

In [9]:
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f"Accuracy на тесте: {accuracy_score(y_test, y_pred):.4f}")

Accuracy на тесте: 0.9504
